# Experiment 1: Generation Metrics

**Experiment:** Exp 1

Compute FID, PSNR, SSIM, and LPIPS metrics for generated images (Tables 5.1-5.3).

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/splits/by_patient_30test1/train.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/trainori_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/trainori.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/trainori.zip" -d "/content/localdata/trainori_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/trainori_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/trainori_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/datasetinge/trainreducido.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/train_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/train.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/train.zip" -d "/content/localdata/train_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/train_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/train_unzipped" | head -n 50

In [ ]:
!pip -q install torch-fidelity

import pandas as pd
from pathlib import Path
import re
import torch
from torch_fidelity import calculate_metrics

# Define the root directory for real images
REAL_IMAGES_ROOT = Path("/content/localdata/trainori_unzipped")

# Define the list of pre-generated synthetic image directories
SYNTHETIC_BATCH_ROOTS_AND_PSIS = [
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi0.90", 0.90),
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi1.00", 1.00),
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi1.10", 1.10),
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi1.20", 1.20),
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi1.30", 1.30),
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi1.40", 1.40)
]

def compute_fid_dir(real_dir: str, gen_dir: str, batch_size: int = 32) -> float:
    metrics = calculate_metrics(
        input1=real_dir,
        input2=gen_dir,
        cuda=torch.cuda.is_available(),
        fid=True,
        kid=False,
        isc=False,
        verbose=False,
        batch_size=batch_size,
        samples_find_deep=True
    )
    return float(metrics["frechet_inception_distance"])

# Get class names from the real images root (assuming it has class subdirectories)
class_names = sorted([d.name for d in REAL_IMAGES_ROOT.iterdir() if d.is_dir()])
print(f"Detected classes in real image root: {class_names}")

fid_results = []

print(f"Calculating FID against real images in: {REAL_IMAGES_ROOT}")

for synth_path_str, psi_value in SYNTHETIC_BATCH_ROOTS_AND_PSIS:
    synth_path = Path(synth_path_str)

    if not synth_path.exists():
        print(f"[WARNING] Synthetic path not found: {synth_path_str}. Skipping.")
        continue

    print(f"\n===== Calculating FID for psi={psi_value} (from {synth_path.name}) =====")

    result_row = {"truncation_psi": psi_value, "synthetic_path": synth_path_str}

    try:
        # Calculate Global FID
        fid_global = compute_fid_dir(str(REAL_IMAGES_ROOT), str(synth_path))
        result_row["fid_global"] = fid_global
        print(f"  -> Global FID for psi={psi_value}: {fid_global:.3f}")

        # Calculate Per-Class FID
        for cls_name in class_names:
            real_cls_dir = REAL_IMAGES_ROOT / cls_name
            gen_cls_dir = synth_path / cls_name
            # Check if both real and generated class directories exist and are not empty
            if real_cls_dir.exists() and gen_cls_dir.exists() and len(list(real_cls_dir.iterdir())) > 0 and len(list(gen_cls_dir.iterdir())) > 0:
                fid_per_cls = compute_fid_dir(str(real_cls_dir), str(gen_cls_dir))
                result_row[f"fid_{cls_name}"] = fid_per_cls
                print(f"    -> FID {cls_name}: {fid_per_cls:.3f}")
            else:
                print(f"    [WARNING] Skipping per-class FID for '{cls_name}' (missing or empty directories).")
                result_row[f"fid_{cls_name}"] = None

    except Exception as e:
        print(f"  [ERROR] Could not compute FID for psi={psi_value}: {e}")
        # Mark all FIDs for this truncation as None if a global error occurs
        result_row["fid_global"] = None
        for cls_name in class_names:
            result_row[f"fid_{cls_name}"] = None

    fid_results.append(result_row)

# --- Display Results ---
print("\n===== FID Results Summary =====")
df_fid_summary = pd.DataFrame(fid_results)
display(df_fid_summary)

In [ ]:
!pip -q install torch-fidelity

import pandas as pd
from pathlib import Path
import re
import torch
from torch_fidelity import calculate_metrics

# Define the root directory for real images
REAL_IMAGES_ROOT = Path("/content/localdata/trainori_unzipped")

# Define the list of pre-generated synthetic image directories
SYNTHETIC_BATCH_ROOTS_AND_PSIS = [
    ("/content/drive/MyDrive/TESIS/synth_batches/N400_psi1.30", 1.30)

]

def compute_fid_dir(real_dir: str, gen_dir: str, batch_size: int = 32) -> float:
    metrics = calculate_metrics(
        input1=real_dir,
        input2=gen_dir,
        cuda=torch.cuda.is_available(),
        fid=True,
        kid=False,
        isc=False,
        verbose=False,
        batch_size=batch_size,
        samples_find_deep=True
    )
    return float(metrics["frechet_inception_distance"])

# Get class names from the real images root (assuming it has class subdirectories)
class_names = sorted([d.name for d in REAL_IMAGES_ROOT.iterdir() if d.is_dir()])
print(f"Detected classes in real image root: {class_names}")

fid_results = []

print(f"Calculating FID against real images in: {REAL_IMAGES_ROOT}")

for synth_path_str, psi_value in SYNTHETIC_BATCH_ROOTS_AND_PSIS:
    synth_path = Path(synth_path_str)

    if not synth_path.exists():
        print(f"[WARNING] Synthetic path not found: {synth_path_str}. Skipping.")
        continue

    print(f"\n===== Calculating FID for psi={psi_value} (from {synth_path.name}) =====")

    result_row = {"truncation_psi": psi_value, "synthetic_path": synth_path_str}

    try:
        # Calculate Global FID
        fid_global = compute_fid_dir(str(REAL_IMAGES_ROOT), str(synth_path))
        result_row["fid_global"] = fid_global
        print(f"  -> Global FID for psi={psi_value}: {fid_global:.3f}")

        # Calculate Per-Class FID
        for cls_name in class_names:
            real_cls_dir = REAL_IMAGES_ROOT / cls_name
            gen_cls_dir = synth_path / cls_name
            # Check if both real and generated class directories exist and are not empty
            if real_cls_dir.exists() and gen_cls_dir.exists() and len(list(real_cls_dir.iterdir())) > 0 and len(list(gen_cls_dir.iterdir())) > 0:
                fid_per_cls = compute_fid_dir(str(real_cls_dir), str(gen_cls_dir))
                result_row[f"fid_{cls_name}"] = fid_per_cls
                print(f"    -> FID {cls_name}: {fid_per_cls:.3f}")
            else:
                print(f"    [WARNING] Skipping per-class FID for '{cls_name}' (missing or empty directories).")
                result_row[f"fid_{cls_name}"] = None

    except Exception as e:
        print(f"  [ERROR] Could not compute FID for psi={psi_value}: {e}")
        # Mark all FIDs for this truncation as None if a global error occurs
        result_row["fid_global"] = None
        for cls_name in class_names:
            result_row[f"fid_{cls_name}"] = None

    fid_results.append(result_row)

# --- Display Results ---
print("\n===== FID Results Summary =====")
df_fid_summary = pd.DataFrame(fid_results)
display(df_fid_summary)

In [ ]:
import os
!pip install gdown --upgrade

if os.path.isdir("/content/drive/MyDrive/colab-sg2-ada-pytorch"):
    %cd "/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch"
elif os.path.isdir("/content/drive/"):
    #install script
    %cd "/content/drive/MyDrive/"
    !mkdir colab-sg2-ada-pytorch
    %cd colab-sg2-ada-pytorch
    !git clone https://github.com/JunTierSS/stylegan2-ada
    %cd stylegan2-ada-pytorch
    !mkdir downloads
    !mkdir datasets
    !mkdir pretrained
    !gdown --id 1-5xZkD8ajXw1DdopTkH_rAoCsD72LhKU -O /content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/pretrained/wikiart.pkl
else:
    !git clone https://github.com/JunTierSS/stylegan2-ada
    %cd stylegan2-ada-pytorch
    !mkdir downloads
    !mkdir datasets
    !mkdir pretrained
    %cd pretrained
    !gdown --id 1-5xZkD8ajXw1DdopTkH_rAoCsD72LhKU
    %cd ../

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/splits/by_patient_30test1/train.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/trainori_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/trainori.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/trainori.zip" -d "/content/localdata/trainori_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/trainori_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/trainori_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/splits/by_patient_30test1/test.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/testori_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/testori.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/testori.zip" -d "/content/localdata/testori_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/testori_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/testori_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/datasetinge/testreducido.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/test_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/test.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/test.zip" -d "/content/localdata/test_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/test_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/test_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/datasetinge/trainreducido.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/train_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/train.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/train.zip" -d "/content/localdata/train_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/train_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/train_unzipped" | head -n 50

In [ ]:
pip install ninja

In [ ]:
pip install opensimplex

In [ ]:

import os, re, io, json, time, argparse, random, csv, shutil, pickle
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import torchvision.transforms.functional as TF
import torchvision

from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
!pip -q install lpips scikit-image pandas

In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import re
import numpy as np
import pandas as pd
from PIL import Image

import torch
import lpips
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def is_img(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def load_img_gray01(path: Path, size: int = 256) -> np.ndarray:
    """Grayscale -> float32 [0,1], shape [H,W,1]."""
    img = Image.open(path).convert("L").resize((size, size), Image.BICUBIC)
    arr = np.array(img).astype(np.float32) / 255.0
    return arr[..., None]

def gray_to_rgb(x_hw1: np.ndarray) -> np.ndarray:
    return np.repeat(x_hw1, 3, axis=2)

def to_lpips_tensor(rgb01: np.ndarray) -> torch.Tensor:
    """[0,1] RGB -> [-1,1] tensor [1,3,H,W]."""
    arr = rgb01 * 2.0 - 1.0
    t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0)
    return t

def compute_metrics_pair(real_rgb01, fake_rgb01, lpips_fn, device):
    psnr = float(peak_signal_noise_ratio(real_rgb01, fake_rgb01, data_range=1.0))
    ssim = float(structural_similarity(real_rgb01, fake_rgb01, channel_axis=2, data_range=1.0))
    with torch.no_grad():
        rt = to_lpips_tensor(real_rgb01).to(device)
        ft = to_lpips_tensor(fake_rgb01).to(device)
        lp = float(lpips_fn(rt, ft).item())
    return psnr, ssim, lp

In [ ]:
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# Always keep group names consistent with the function below
REAL_RX = re.compile(
    r"^real_(?P<real>[a-zA-Z0-9\-]+)\.(png|jpg|jpeg|bmp|tif|tiff)$",
    re.IGNORECASE
)
CF_RX = re.compile(
    r"^cf_to_(?P<gen>[a-zA-Z0-9\-]+)_i\d+_f\d+\.(png|jpg|jpeg|bmp|tif|tiff)$",
    re.IGNORECASE
)

def is_img(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def find_real_and_cfs(patient_dir: Path) -> Tuple[Optional[Path], Optional[str], Dict[str, List[Path]]]:
    """
    Returns:
      real_path, real_class, {gen_class: [cf_paths...]}
    """
    real_path, real_class = None, None
    cfs: Dict[str, List[Path]] = {}

    for p in patient_dir.iterdir():
        if not (p.is_file() and is_img(p)):
            continue

        m_real = REAL_RX.match(p.name)
        if m_real:
            real_path = p
            real_class = m_real.group("real").lower()
            continue

        m_cf = CF_RX.match(p.name)
        if m_cf:
            # robust: take named group if present, else take first capture
            if "gen" in m_cf.groupdict():
                gen_class = m_cf.group("gen").lower()
            else:
                gen_class = m_cf.group(1).lower()
            cfs.setdefault(gen_class, []).append(p)

    return real_path, real_class, cfs

In [ ]:
def compute_metrics_from_classification(
    classification_root: Path,
    class_names: List[str],
    image_size: int = 256,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    device = "cuda" if torch.cuda.is_available() else "cpu"
    lpips_fn = lpips.LPIPS(net="alex").to(device).eval()

    class_lower = [c.lower() for c in class_names]

    rows = []
    patient_dirs = [d for d in classification_root.iterdir() if d.is_dir()]
    print(f"Found {len(patient_dirs)} patient folders.")

    for pd_dir in patient_dirs:
        real_p, real_cls, cfs = find_real_and_cfs(pd_dir)
        if real_p is None or real_cls is None:
            continue
        if real_cls not in class_lower:
            continue

        real_gray = load_img_gray01(real_p, size=image_size)
        real_rgb  = gray_to_rgb(real_gray)

        for gen_cls, cf_list in cfs.items():
            if gen_cls not in class_lower:
                continue

            # si hay múltiples cf_to para la misma clase (distintos i/f),
            # calculamos métricas para todos y promediamos luego
            for cf_p in cf_list:
                fake_gray = load_img_gray01(cf_p, size=image_size)
                fake_rgb  = gray_to_rgb(fake_gray)

                psnr, ssim, lp = compute_metrics_pair(real_rgb, fake_rgb, lpips_fn, device)

                rows.append({
                    "patient_dir": str(pd_dir),
                    "real_class": real_cls,
                    "gen_class": gen_cls,
                    "real_path": str(real_p),
                    "gen_path": str(cf_p),
                    "psnr": psnr,
                    "ssim": ssim,
                    "lpips": lp,
                })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError("No pairs found. Check filename patterns or class_names.")

    # Tabla 1: diagonal por clase real (real==gen)
    diag = df[df["real_class"] == df["gen_class"]]
    tab1 = (
        diag.groupby("real_class")[["psnr","ssim","lpips"]]
            .agg(["mean","std"])
            .reset_index()
    )

    # Tabla 2: 9 combinaciones (promedio por real×gen)
    tab2 = (
        df.groupby(["real_class","gen_class"])[["psnr","ssim","lpips"]]
          .mean()
          .reset_index()
    )

    return df, tab1, tab2

In [ ]:
classification_root = Path("/content/localdata/class/classification")
class_names = ["glioma", "meningioma", "pituitary"]  # ajusta si corresponde

df_all, tab1_per_class, tab2_9combos = compute_metrics_from_classification(
    classification_root=classification_root,
    class_names=class_names,
    image_size=256
)

tab1_per_class

In [ ]:
# Assume you already have df_all and tab1_per_class from compute_metrics_from_classification(...)

metrics_cols = ["psnr", "ssim", "lpips"]

# 1) Take only diagonal pairs
diag = df_all[df_all["real_class"] == df_all["gen_class"]]

# 2) Compute overall mean/std across ALL diagonal samples
total_stats = diag[metrics_cols].agg(["mean", "std"])

# 3) Build a single-row DataFrame with the same MultiIndex columns as tab1_per_class
total_row = pd.DataFrame(
    { (m, stat): total_stats.loc[stat, m]
      for m in metrics_cols for stat in ["mean", "std"] },
    index=["total"]
).reset_index().rename(columns={"index": "real_class"})

# 4) Append to the per-class table
tab1_per_class_total = pd.concat([tab1_per_class, total_row], ignore_index=True)

tab1_per_class_total

In [ ]:
tab2_9combos

In [ ]:
!pip -q install torch-fidelity

In [ ]:
from pathlib import Path
import re, shutil
from collections import defaultdict
import torch
from torch_fidelity import calculate_metrics

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

REAL_RX = re.compile(r"^real_(?P<cls>[a-zA-Z0-9\-]+)\.(png|jpg|jpeg|bmp|tif|tiff)$", re.IGNORECASE)
CF_RX   = re.compile(r"^cf_to_(?P<cls>[a-zA-Z0-9\-]+)_(?:i\d+_f\d+)\.(png|jpg|jpeg|bmp|tif|tiff)$", re.IGNORECASE)

def is_img(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def scan_classification_root(classification_root: Path):
    """
    Returns:
      real_by_class: dict cls -> list(real_paths)
      gen_by_class:  dict cls -> list(cf_paths)
    """
    real_by_class = defaultdict(list)
    gen_by_class  = defaultdict(list)

    patient_dirs = [d for d in classification_root.iterdir() if d.is_dir()]
    print(f"Found {len(patient_dirs)} patient folders.")

    for pd_dir in patient_dirs:
        for p in pd_dir.iterdir():
            if not (p.is_file() and is_img(p)):
                continue

            m_real = REAL_RX.match(p.name)
            if m_real:
                cls = m_real.group("cls").lower()
                real_by_class[cls].append(p)
                continue

            m_cf = CF_RX.match(p.name)
            if m_cf:
                cls = m_cf.group("cls").lower()
                gen_by_class[cls].append(p)

    # Keep only classes that exist in both sets
    classes = sorted(set(real_by_class.keys()) & set(gen_by_class.keys()))
    real_by_class = {c: real_by_class[c] for c in classes}
    gen_by_class  = {c: gen_by_class[c]  for c in classes}

    print("Classes found:", classes)
    for c in classes:
        print(f"  {c:12s} real={len(real_by_class[c])}  gen={len(gen_by_class[c])}")

    return real_by_class, gen_by_class, classes

In [ ]:
def prepare_fid_folders(real_by_class, gen_by_class, classes, tmp_root="/content/fid_tmp"):
    """
    Creates:
      tmp_root/real/<class>/*.png
      tmp_root/gen/<class>/*.png
    Copies images from Drive into /content for fast FID.
    """
    tmp_root = Path(tmp_root)
    if tmp_root.exists():
        shutil.rmtree(tmp_root)
    (tmp_root / "real").mkdir(parents=True, exist_ok=True)
    (tmp_root / "gen").mkdir(parents=True, exist_ok=True)

    for cls in classes:
        real_out = tmp_root / "real" / cls
        gen_out  = tmp_root / "gen" / cls
        real_out.mkdir(parents=True, exist_ok=True)
        gen_out.mkdir(parents=True, exist_ok=True)

        # copy reals
        for i, p in enumerate(real_by_class[cls]):
            shutil.copyfile(p, real_out / f"real_{cls}_{i:05d}{p.suffix.lower()}")

        # copy gens (all cf_to_<cls> variants)
        for i, p in enumerate(gen_by_class[cls]):
            shutil.copyfile(p, gen_out / f"gen_{cls}_{i:05d}{p.suffix.lower()}")

    return tmp_root

In [ ]:
def compute_fid_dir(real_dir: str, gen_dir: str, batch_size: int = 32) -> float:
    metrics = calculate_metrics(
        input1=real_dir,
        input2=gen_dir,
        cuda=torch.cuda.is_available(),
        fid=True,
        kid=False,
        isc=False,
        verbose=False,
        batch_size=batch_size,
        samples_find_deep=True
    )
    return float(metrics["frechet_inception_distance"])


# ===== RUN =====
classification_root = Path("/content/localdata/class/classification")

real_by_class, gen_by_class, classes = scan_classification_root(classification_root)
tmp_root = prepare_fid_folders(real_by_class, gen_by_class, classes, tmp_root="/content/fid_tmp")

# Global FID (all classes together)
fid_global = compute_fid_dir(str(tmp_root/"real"), str(tmp_root/"gen"), batch_size=32)
print("\nFID global:", fid_global)

# Per-class FID (diagonal)
fid_per_class = {}
for cls in classes:
    fid_per_class[cls] = compute_fid_dir(
        str(tmp_root/"real"/cls),
        str(tmp_root/"gen"/cls),
        batch_size=32
    )

print("\nFID per class:")
for k,v in fid_per_class.items():
    print(f"  {k:12s}: {v:.3f}")